In [ ]:
import arviz as az
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import Markdown
from matplotlib.pyplot import cm
import numpy as np

from emu_renewal.inputs import ANALYSIS_TYPES, get_latest_analyses
from emu_renewal.outputs import get_output_dir, melt_df_except_first_level, get_multianalysis_dispvals_from_idatas, get_multianalysis_procvals_from_idatas, get_multianalysis_likelihoods
from emu_renewal.outputs import get_multianalysis_ind_spaghetti, get_multianalysis_procvals
from emu_renewal.plotting import plot_process_comparison, plot_updates_comparison, plot_mob_update_comparison

set_matplotlib_formats("svg")

In [ ]:
countries = ["France", "Sweden", "Portugal", "Spain"]
analysis_times = {c: get_latest_analyses(c, ANALYSIS_TYPES) for c in countries}
n_countries = len(countries)

In [ ]:
idatas = {}
proc_dfs = {}
disp_dfs = {}
likelihoods = {}
for country in countries:
    country_analysis_times = analysis_times[country]
    country_idatas = {k: az.from_netcdf(get_output_dir(country, k, v) / "idata.nc") for k, v in country_analysis_times.items()}
    idatas[country] = country_idatas
    proc_dfs[country] = melt_df_except_first_level(get_multianalysis_procvals_from_idatas(country_idatas))
    disp_dfs[country] = melt_df_except_first_level(get_multianalysis_dispvals_from_idatas(country_idatas))
    likelihoods[country] = melt_df_except_first_level(get_multianalysis_likelihoods(country, analysis_times[country]))

In [ ]:
disp_fig, axes = plt.subplots(1, n_countries, figsize=(6 * n_countries, 6))
for c, country in enumerate(countries):
    c_ax = sns.kdeplot(disp_dfs[country], fill=True, ax=axes[c])
    c_ax.set_title(country)
    c_ax.set_yticks([])
disp_fig.savefig("disp_fig.svg")

In [ ]:
#| fig-cap: "Comparison of likelihoods by inclusion of candidate mobility modifiers."
like_fig, axes = plt.subplots(1, n_countries, figsize=(6 * n_countries, 6))
for c, country in enumerate(countries):
    c_ax = sns.kdeplot(likelihoods[country], fill=True, ax=axes[c])
    c_ax.set_title(country)
    c_ax.set_yticks([])
like_fig.savefig("like_fig.svg")

In [ ]:
cm.Accent.colors[0]
proc_fig, axes = plt.subplots(1, n_countries, figsize=(6 * n_countries, 6))
for c, country in enumerate(countries):
    spagh_df = get_multianalysis_ind_spaghetti(country, "process", analysis_times[country])
    for a, analysis in enumerate(analysis_times[country]):
        data = spagh_df[analysis].quantile([0.05, 0.5, 0.95], axis=1).T
        colour = cm.Accent.colors[a]
        axes[c].plot(data.index, data[0.5], color=colour, label=analysis, linewidth=3.0)
        axes[c].plot(data.index, data[[0.05, 0.95]], color=colour, linestyle="--", linewidth=0.5)
        axes[c].set_title(country)
        axes[c].legend()
        plt.setp(axes[c].xaxis.get_majorticklabels(), rotation=70)

proc_fig.savefig("proc_fig.svg")

In [ ]:
#| tbl-cap: "Median likelihood values by inclusion of candidate mobility modifiers."
medians = likelihoods.median()
medians.index.name = "Analysis type"
medians.name = "Median values"
Markdown(medians.to_markdown())